# Two-layer MLP with Manual Backpropagation

Architecture:

`input -> linear1 -> relu -> linear2 -> softmax cross-entropy loss`

This notebook uses the NumPy implementation in `experiments/chapter01/two_layer_mlp_numpy.py`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "chapter01":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from experiments.chapter01.two_layer_mlp_numpy import (
    backward,
    cross_entropy_loss,
    evaluate,
    forward,
    gradient_check,
    init_params,
    make_dataset,
    train,
)

np.set_printoptions(precision=4, suppress=True)

## Dataset

Use a normalized two-class `make_moons` dataset. The input dimension is 2 and the output class count is 2.

In [ ]:
x_train, y_train, x_val, y_val = make_dataset()
print(x_train.shape, y_train.shape, x_val.shape, y_val.shape)
print("classes:", np.unique(y_train))

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, s=12, cmap="coolwarm", alpha=0.75)
plt.title("Training data")
plt.xlabel("x0")
plt.ylabel("x1")
plt.show()

## Forward Pass Shape Check

`x @ w1 + b1 -> z1 -> relu -> a1 -> a1 @ w2 + b2 -> logits -> softmax -> probs`

In [ ]:
params = init_params(input_dim=2, hidden_dim=32, output_dim=2)
cache = forward(x_train[:8], params)

print("x      ", cache.x.shape)
print("w1     ", params.w1.shape)
print("b1     ", params.b1.shape)
print("z1     ", cache.z1.shape)
print("a1     ", cache.a1.shape)
print("w2     ", params.w2.shape)
print("b2     ", params.b2.shape)
print("logits ", cache.logits.shape)
print("probs  ", cache.probs.shape)
print("row sums:", cache.probs.sum(axis=1))
print("loss:", cross_entropy_loss(cache.probs, y_train[:8]))

## Backward Pass Shape Check

For softmax + cross entropy, the gradient entering `linear2` is:

`dlogits = (probs - one_hot(y)) / batch_size`

In [ ]:
grads = backward(cache, y_train[:8], params)
print("dw1", grads.w1.shape, "matches", params.w1.shape)
print("db1", grads.b1.shape, "matches", params.b1.shape)
print("dw2", grads.w2.shape, "matches", params.w2.shape)
print("db2", grads.b2.shape, "matches", params.b2.shape)
print("gradient check max abs error:", gradient_check())

## Train

In [ ]:
params, history = train(
    x_train,
    y_train,
    x_val,
    y_val,
    hidden_dim=32,
    epochs=400,
    batch_size=64,
    lr=0.08,
    log_every=50,
)

print("final train:", evaluate(x_train, y_train, params))
print("final val:", evaluate(x_val, y_val, params))

In [ ]:
epochs = np.arange(1, len(history.train_loss) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, history.train_loss, label="train")
axes[0].plot(epochs, history.val_loss, label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, history.train_acc, label="train")
axes[1].plot(epochs, history.val_acc, label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.show()

## Decision Boundary

In [ ]:
x_min, x_max = x_train[:, 0].min() - 0.5, x_train[:, 0].max() + 0.5
y_min, y_max = x_train[:, 1].min() - 0.5, x_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
probs = forward(grid, params).probs[:, 1].reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, probs, levels=30, cmap="coolwarm", alpha=0.55)
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, s=12, cmap="coolwarm", edgecolors="none")
plt.title("Decision boundary")
plt.xlabel("x0")
plt.ylabel("x1")
plt.show()